## 💻 SFT for Bash Suggestion
*Experiment: Best run*

---

### 📌 Key Parameters
*   **Rank:** `32`
*   **Alpha:** `32`
*   **Target Gates:** `q`, `k`, `v`, `o`, `gate`, `up`, `down`
*   **Epochs:** `2`
*   **Learning Rate:** `5e-4`

## 1) Import Libraries

In [1]:
# Check GPU
!nvidia-smi

Sun Oct  4 10:53:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:81:00.0 Off |                  N/A |
|  0%   49C    P8             30W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import unsloth
import os
import gc
from pathlib import Path

import torch
from unsloth import FastLanguageModel
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, DatasetDict
from trl import SFTTrainer, SFTConfig

from huggingface_hub import login, HfApi, upload_folder
import getpass
import datetime

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1004 10:54:01.740000 1358 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1004 10:54:01.958000 1358 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:3525: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


## 2) Config (model + data + hyperparams)


In [3]:
set_seed(42)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_NAME = "jiacheng-ye/nl2bash"
SYSTEM_PROMPT = """
You are an expert Linux command-line assistant. Your sole task is to translate natural language requests into precise, executable Bash commands.

Strict Output Rules:
1. Output ONLY the raw Bash command.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```bash ... ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the command:" or "Sure!").
4. Chain commands logically using pipes (|), &&, or || where appropriate.
"""


# ====== TRAINING ======
MAX_SEQ_LENGTH = 2048
RANK = 32
ALPHA_RANK = 32
GATES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
NUM_EPOCHS = 2
LR = 5e-4
WARMUP_STEPS = 100

# Batch
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8

# Outputs
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "qwen2.5-bash-7B-best-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Memory
LOAD_IN_4BIT = False

# dtype
DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("BASE_MODEL:", BASE_MODEL)
print("DTYPE:", DTYPE, "| LOAD_IN_4BIT:", LOAD_IN_4BIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


BASE_MODEL: Qwen/Qwen2.5-Coder-7B-Instruct
DTYPE: bfloat16 | LOAD_IN_4BIT: False
GPU: NVIDIA GeForce RTX 3090


## 3) Load model + Turn on LoRA (Unsloth)


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = RANK,
    target_modules = GATES,
    lora_alpha = ALPHA_RANK,
    bias = "none",
    use_gradient_checkpointing = True,
    use_rslora = False,
    loftq_config = None,
    random_state=42
)
model.config.use_cache = False
model.print_trainable_parameters()

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Loaded model + LoRA OK")


==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.561 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/huggingface_hub/constants.py:301: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Loading weights: 100%|██████████| 339/339 [00:02<00:00, 146.69it/s]
Unsloth 2026.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491
Loaded model + LoRA OK


## 4) Load dataset JSONL & Format by chat_template


In [5]:
raw_dataset = load_dataset(DATASET_NAME, revision="refs/convert/parquet")
dataset = DatasetDict({
    "train": raw_dataset["train"],
    "val": raw_dataset["validation"],
    "test": raw_dataset["test"]
})

print(dataset)
sample = dataset["train"][0]
print("\nQuestion preview:\n")
print(sample["nl"][:500])
print("\nAnswer preview:\n")
print(sample["bash"][:500])

Generating train split: 8090 examples [00:00, 456276.57 examples/s]
Generating validation split: 609 examples [00:00, 168114.46 examples/s]
Generating test split: 606 examples [00:00, 188851.19 examples/s]

DatasetDict({
    train: Dataset({
        features: ['nl', 'bash'],
        num_rows: 8090
    })
    val: Dataset({
        features: ['nl', 'bash'],
        num_rows: 609
    })
    test: Dataset({
        features: ['nl', 'bash'],
        num_rows: 606
    })
})

Question preview:

Do a dry run of renaming file extension '.andnav' to '.tile' for all files/directories under current directory tree

Answer preview:

find . -name "*.andnav" | rename -vn "s/\.andnav$/.tile/"


In [6]:
def formatting_prompts_func(examples):
    questions = examples["nl"]
    answers = examples["bash"]

    texts = []

    for q, a in zip(questions, answers):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q.strip()},
            {"role": "assistant", "content": a.strip()}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)

Map (num_proc=2): 100%|██████████| 606/606 [00:00<00:00, 1619.57 examples/s]


In [7]:
print("Dataset size:", len(dataset["train"]))
print("\n===== PREVIEW 2 SAMPLES =====")
for i in range(2):
    print(f"\n--- Sample {i+1} ---\n{dataset['train'][i]['text']}\n...")

Dataset size: 8090

===== PREVIEW 2 SAMPLES =====

--- Sample 1 ---
<|im_start|>system

You are an expert Linux command-line assistant. Your sole task is to translate natural language requests into precise, executable Bash commands.

Strict Output Rules:
1. Output ONLY the raw Bash command.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```bash ... ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the command:" or "Sure!").
4. Chain commands logically using pipes (|), &&, or || where appropriate.
<|im_end|>
<|im_start|>user
Do a dry run of renaming file extension '.andnav' to '.tile' for all files/directories under current directory tree<|im_end|>
<|im_start|>assistant
find . -name "*.andnav" | rename -vn "s/\.andnav$/.tile/"<|im_end|>

...

--- Sample 2 ---
<|im_start|>system

You are an expert Linux command-line assistant. Your sole task is to translate natural language requests into precise, executab

## 5) Train SFT


In [8]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,

    args = SFTConfig(
        gradient_accumulation_steps = GRAD_ACCUM,
        per_device_train_batch_size = PER_DEVICE_BATCH,
        per_device_eval_batch_size = PER_DEVICE_BATCH,
        learning_rate = LR,
        num_train_epochs = NUM_EPOCHS,
        warmup_steps = WARMUP_STEPS,

        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",

        save_strategy = "steps",
        save_steps = 20,
        save_total_limit = 2,

        fp16 = (DTYPE == "float16"),
        bf16 = (DTYPE == "bfloat16"),
        optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = OUTPUT_DIR,

        completion_only_loss = True,
        gradient_checkpointing=True
    ),
)

train_result = trainer.train()
train_result.metrics

Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 609/609 [00:04<00:00, 135.81 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,090 | Num Epochs = 2 | Total steps = 1,012
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/root/ai-backend-assistant/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_

Step,Training Loss,Validation Loss
20,2.194601,0.851103
40,0.526709,0.385923
60,0.353581,0.360240
80,0.330580,0.349058
100,0.341256,0.340350
120,0.323750,0.336238
140,0.311214,0.332864
160,0.324108,0.328661
180,0.315493,0.324645
200,0.317414,0.323396


Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/checkpoint-80/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.

{'train_runtime': 5500.5675,
 'train_samples_per_second': 2.942,
 'train_steps_per_second': 0.184,
 'total_flos': 1.1043924552544666e+17,
 'train_loss': 0.30464352213818097,
 'epoch': 2.0}

In [9]:
for callback in trainer.callback_handler.callbacks.copy():
    if "Notebook" in callback.__class__.__name__:
        trainer.remove_callback(callback)

eval_results = trainer.evaluate()
print("===== VAL SET METRICS =====")
eval_results

===== VAL SET METRICS =====


{'eval_loss': 0.28173828125,
 'eval_runtime': 38.9239,
 'eval_samples_per_second': 15.646,
 'eval_steps_per_second': 7.836,
 'epoch': 2.0}

In [10]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_test_dataset = dataset["test"].map(
    tokenize_function,
    batched=True,
    num_proc=2
)

test_results = trainer.evaluate(
    eval_dataset=tokenized_test_dataset,
    metric_key_prefix="test"
)

print("===== TEST SET METRICS =====")
print(test_results)

Map (num_proc=2): 100%|██████████| 606/606 [00:00<00:00, 982.00 examples/s]


===== TEST SET METRICS =====
{'test_loss': 0.2756209075450897, 'test_runtime': 39.1338, 'test_samples_per_second': 15.485, 'test_steps_per_second': 7.743, 'epoch': 2.0}


## 6) Save LoRA Adapter

In [11]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora/tokenizer_config.json.


Saved LoRA adapter to: /root/ai-backend-assistant/bash-suggester/notebooks/artifacts/qwen2.5-bash-7B-best-lora


## 7) Push to Hugging Face

In [12]:
hf_token = getpass.getpass()
login(token=hf_token)
print("HF login OK")

 ········


HF login OK


In [13]:
api = HfApi()
USERNAME = api.whoami()["name"]

REPO = f"{USERNAME}/Qwen2.5-Coder-7B-Instruct-bash-LoRA"
api.create_repo(REPO, private=False, exist_ok=True)

RepoUrl('https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-bash-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='hn-minh/Qwen2.5-Coder-7B-Instruct-bash-LoRA')

In [26]:
ckpt_path = "./artifacts/qwen2.5-bash-7B-best-lora"
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = AutoModelForCausalLM.from_pretrained(ckpt_path)

Loading weights: 100%|██████████| 392/392 [00:00<00:00, 1200.61it/s]


In [28]:
model.push_to_hub(
    REPO,
    commit_message="Add model ckpt",
)

tokenizer.push_to_hub(
    REPO,
    commit_message="Add tokenizer ckpt",
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          |  606kB /  162MB, 58.2kB/s  
Processing Files (0 / 1):   1%|          | 1.21MB /  162MB,  114kB/s  
Processing Files (0 / 1):   2%|▏         | 2.42MB /  162MB,  227kB/s  
Processing Files (0 / 1):   3%|▎         | 4.24MB /  162MB,  397kB/s  
Processing Files (0 / 1):   4%|▍         | 6.06MB /  162MB,  564kB/s  
Processing Files (0 / 1):   5%|▌         | 8.48MB /  162MB,  787kB/s  
Processing Files (0 / 1):   6%|▌         | 9.10MB /  162MB,  836kB/s  
Processing Files (0 / 1):   8%|▊         | 12.1MB /  162MB, 1.10MB/s  
Processing Files (0 / 1):   9%|▉         | 14.6MB /  162MB, 1.32MB/s  
Processing Files (0 / 1):  11%|█▏        | 18.2MB /  162MB, 1.64MB/s  
Processing Files (0 / 1):  12%|█▏        | 18.8MB /  162MB, 1.69MB/s  
Processing Files (0 / 1):  13%|█▎        | 21.2MB /  162MB, 1.89MB/s  
Processing Fi

CommitInfo(commit_url='https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-bash-LoRA/commit/ddacdc352415cfda9984198a2c6fd3089b057f1b', commit_message='Add tokenizer ckpt', commit_description='', oid='ddacdc352415cfda9984198a2c6fd3089b057f1b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hn-minh/Qwen2.5-Coder-7B-Instruct-bash-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='hn-minh/Qwen2.5-Coder-7B-Instruct-bash-LoRA'), pr_revision=None, pr_num=None)

## 8) Sanity check


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
model.to(DEVICE)
FastLanguageModel.for_inference(model)

user_question = "Write a FastAPI endpoint (async def) that accepts a long piece of text, uses Hugging Face AutoTokenizer to count the number of tokens, and returns the result. The system needs to handle thousands of concurrent requests."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(DEVICE)

outputs = model.generate(
    **inputs,
    max_new_tokens=3000,
    use_cache=True,
    temperature=0.1,
)

prompt_length = inputs["input_ids"].shape[1]
response_tokens = outputs[0][prompt_length:]
response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("===== TEST INFERENCE =====")
print(f"User: {user_question}")
print(f"Code: {response.strip()}")